In [1]:
import hanoi
import cProfile
import pstats
import numpy as np
import scipy
import jax.numpy as jnp
import math

In [2]:
%load_ext Cython

In [ ]:

print(np.dtype(int))      # shows default integer type
print(np.dtype(np.int32)) # 32-bit
print(np.dtype(np.int64)) # 64-bit

In [ ]:
%%cython --annotate
import numpy as np
cimport numpy as cnp
cimport cython

def sim_reproduction_c(int popsize, cnp.float64_t[:, :] genotype, cnp.float64_t[:] fitness):

    cdef int n_loci = genotype.shape[0]
    cdef cnp.float64_t[:, :] genotype_next
    cdef cnp.int64_t[:] n_offspring

    # when no individuals reproduce
    if np.sum(fitness) == 0:
        genotype_next = np.full((n_loci, popsize), np.nan)
        n_offspring = np.zeros(popsize, dtype = int)
        return np.asarray(genotype_next), np.asarray(n_offspring)

    # When at least one individuals reproduces
    # Total fitness
    #cdef cnp.float64_t fitness_sum = np.sum(fitness)
    # Declare variables
    cdef cnp.float64_t[:] pvals
    cdef cnp.int64_t[:] sam
    cdef int i, j, k, idx
    cdef cnp.float64_t bit1, bit2
    cdef cnp.float64_t fitness_sum = 0
    for i in range(popsize):
        fitness_sum += fitness[i]
    #cdef cnp.int64_t[:,:] genotype_next_T = np.empty(genotype.shape, dtype = np.int64)
    #cdef int i
    # compute binomial parameter for each individual
    pvals = np.empty(fitness.shape[0], dtype=np.float64)
    for i in range(fitness.shape[0]):
        pvals[i] = fitness[i] / fitness_sum
    n_offspring = np.random.multinomial(n = 2 * popsize, pvals = pvals) 
    # index of 2N parents
    sam = np.repeat(range(popsize), n_offspring)
    np.random.shuffle(sam)
    # Make genotype of next parents randomly take one haplotype per locus per parent
    genotype_next = np.empty((n_loci, popsize), dtype=np.float64)
    for i in range(popsize):
        idx = 2*i
        for j in range(n_loci):
            # sample two haplotypes
            bit1 = np.random.binomial(1, genotype[j, sam[idx]]/2)
            bit2 = np.random.binomial(1, genotype[j, sam[idx+1]]/2)
            genotype_next[j, i] = bit1 + bit2
    return np.asarray(genotype_next), np.asarray(n_offspring)
 







In [286]:
gt = np.array([[0, 1, 2],
               [1, 2, 0],
               [2, 0, 1]], dtype = np.float64)

In [323]:
hanoi.sim_reproduction(3, gt, np.array([0,0,0]))

(array([[nan, nan, nan],
        [nan, nan, nan],
        [nan, nan, nan]]),
 array([0, 0, 0]))

In [92]:
np.array([1, 0, 0], dtype=float)

array([1., 0., 0.])

In [322]:
sim_reproduction_c(3, gt, np.array([0, 0, 0], dtype = np.float64))

(array([[nan, nan, nan],
        [nan, nan, nan],
        [nan, nan, nan]]),
 array([0, 0, 0]))

In [198]:
np.int64

numpy.int64

Test the speed

In [8]:
gt = np.ones((10_000, 1_000))

In [9]:
idx = np.random.choice(np.arange(10_000 * 1_000), size = 5_000_000, replace = True)
rows, cols = np.unravel_index(idx, gt.shape)

In [10]:
gt[rows[0:2_500_000], cols[0:2_500_000]] = 0
gt[rows[2_500_000:5_000_000], cols[2_500_000:5_000_000]] = 2

In [11]:
gt

array([[1., 0., 1., ..., 1., 1., 2.],
       [1., 0., 1., ..., 1., 1., 1.],
       [2., 2., 1., ..., 1., 1., 2.],
       ...,
       [1., 1., 1., ..., 1., 0., 1.],
       [1., 0., 1., ..., 1., 1., 0.],
       [2., 1., 2., ..., 1., 2., 1.]])

In [12]:
fitness = np.random.uniform(0, 1, 1_000)

In [17]:
%time
res = hanoi.sim_reproduction(popsize = 1_000, genotype = gt, fitness = fitness)

CPU times: user 2 μs, sys: 0 ns, total: 2 μs
Wall time: 4.77 μs


In [20]:
%time
res = hanoi.sim_reproduction_c(popsize = 1_000, genotype = gt, fitness = fitness)

CPU times: user 3 μs, sys: 0 ns, total: 3 μs
Wall time: 4.77 μs


In [330]:
np.random.multinomial?


Signature: np.random.multinomial(n, pvals, size=None)
Docstring:
multinomial(n, pvals, size=None)

Draw samples from a multinomial distribution.

The multinomial distribution is a multivariate generalization of the
binomial distribution.  Take an experiment with one of ``p``
possible outcomes.  An example of such an experiment is throwing a dice,
where the outcome can be 1 through 6.  Each sample drawn from the
distribution represents `n` such experiments.  Its values,
``X_i = [X_0, X_1, ..., X_p]``, represent the number of times the
outcome was ``i``.

.. note::
    New code should use the `~numpy.random.Generator.multinomial`
    method of a `~numpy.random.Generator` instance instead;
    please see the :ref:`random-quick-start`.

.. warning::
  This function defaults to the C-long dtype, which is 32bit on windows
  and otherwise 64bit on 64bit platforms (and 32bit on 32bit ones).
  Since NumPy 2.0, NumPy's default integer is 32bit on 32bit platforms
  and 64bit on 64bit platforms.



## sim_generation

In [2]:
%%cython --annotate
import numpy as np
import hanoi
cimport numpy as cnp
cimport cython

cpdef object sim_generation_c(object mut_effect, cnp.ndarray[cnp.float64_t, ndim = 2] genotype, double mut_rate, str fit_func, cnp.ndarray[cnp.float64_t, ndim = 1] mean_0, cnp.ndarray[cnp.float64_t, ndim = 2] cov_0, dict kwargs = {}):
    cdef int popsize = genotype.shape[1]
    
    # Breeding value
    cdef object A = hanoi.comp_breed_val(mut_effect=mut_effect, genotype=genotype)
    # Phenotype
    cdef cnp.ndarray[cnp.float64_t, ndim = 2] z = hanoi.sim_pheno(breed_val=A, mean_0=mean_0, cov_0 = cov_0)
    
    # Phenotype to fitness
    cdef cnp.float64_t[:] w
    cdef object z_opt # because it could be 1D or 2D
    cdef object sigma # because it could be scalar or 1D
    cdef cnp.float64_t[:] p
    cdef cnp.float64_t[:,:,:] boxes
    if fit_func == "fit_gaus":
        z_opt = kwargs["z_opt"]
        sigma = kwargs["sigma"]
        w = hanoi.fit_gaus(z=z, z_opt = z_opt, sigma = sigma)
    if fit_func == "fit_multimodal":
        sigma = kwargs["sigma"]
        p = kwargs["p"]
        z_opt = kwargs["z_opt"]
        w = hanoi.fit_multimodal(sigma=sigma, p=kwargs.get('p'), z_opt=z_opt, z=z)
    if fit_func == "fit_neutral":
        w = hanoi.fit_neutral(z=z)
    if fit_func == "fit_step":
        boxes = kwargs["boxes"]
        w = hanoi.fit_step(z=z, boxes=boxes)
    
    # Reproduction
    cdef cnp.ndarray[cnp.float64_t, ndim = 2] genotype_next
    cdef cnp.ndarray[cnp.int64_t, ndim = 1] n_offspring
    genotype_next, n_offspring = hanoi.sim_reproduction(popsize, genotype, w)
    #if n_offspring.sum() == 0:
    #    raise RuntimeError("No individuals survived")
    genotype_next = hanoi.sim_mutation(genotype=genotype_next, mut_rate=mut_rate) # This still returns nans if it is nans
    return hanoi.Generation(genotype = genotype, 
                      genotype_next = genotype_next,
                      breed_val = A, 
                      phenotype = z, 
                      fitness = w,
                      n_offspring = n_offspring
                     )



UsageError: Cell magic `%%cython` not found.


In [79]:
np.empty((0,)).ndim

1

In [147]:
M = hanoi.MutEffect(mean = np.ones((n, L), dtype = np.float64), 
                    var = np.ones((n, L), dtype = np.float64), 
                    cov = np.zeros((math.comb(n, 2), L), dtype = np.float64)
                   )

In [81]:
M.show()

Number of traits n:
1
Number of trait pairs:
0
Number of loci L:
3
Mutation effect of mean:
[[1. 1. 1.]]
Mutation effect of variance:
[[1. 1. 1.]]
Mutation effect of covariance:
[]


In [82]:
gt = np.ones((L, N), dtype = np.float64)

In [83]:
mean_0 = np.array([0], dtype = float)
varcov_0 = np.array([[0.1]])

In [84]:
gen = sim_generation_c(mut_effect = M, genotype = gt, fit_func = "fit_neutral", mut_rate = 0.0, mean_0 = mean_0, cov_0 = varcov_0)

In [85]:
gen.allele_freqs()

array([[0.5, 0.5, 0.5]])

In [109]:
%%time
for i in range(10000):
    gen = sim_generation_c(mut_effect = M, genotype = gt, fit_func = "fit_gaus", mut_rate = 0.0, mean_0 = mean_0, cov_0 = varcov_0, kwargs = {"z_opt": np.array([0], dtype = float), "sigma": 1})

CPU times: user 797 ms, sys: 778 μs, total: 798 ms
Wall time: 797 ms


In [110]:
%%time
for i in range(10000):
    gen = hanoi.sim_generation(mut_effect = M, genotype = gt, fit_func = "fit_neutral", mut_rate = 0.0, mean_0 = mean_0, cov_0 = varcov_0, z_opt = np.array([0]), sigma = 1)

CPU times: user 629 ms, sys: 5.82 ms, total: 634 ms
Wall time: 632 ms


## sim_generations

In [20]:
%%cython -a -f --compile-args=-DCYTHON_TRACE=1 --link-args=-DCYTHON_TRACE=1
# cython: linetrace=True, binding=True, profile=True
import numpy as np
import hanoi
cimport numpy as cnp
cimport cython

cpdef object sim_generation_c(object mut_effect, cnp.ndarray[cnp.float64_t, ndim = 2] genotype, double mut_rate, str fit_func, cnp.ndarray[cnp.float64_t, ndim = 1] mean_0, cnp.ndarray[cnp.float64_t, ndim = 2] cov_0, dict kwargs = {}):
    cdef int popsize = genotype.shape[1]
    
    # Breeding value
    cdef object A = hanoi.comp_breed_val(mut_effect=mut_effect, genotype=genotype)
    # Phenotype
    cdef cnp.ndarray[cnp.float64_t, ndim = 2] z = hanoi.sim_pheno(breed_val=A, mean_0=mean_0, cov_0 = cov_0)
    
    # Phenotype to fitness
    cdef cnp.ndarray[cnp.float64_t, ndim=1] w
    cdef object z_opt # because it could be 1D or 2D
    cdef object sigma # because it could be scalar or 1D
    cdef cnp.float64_t[:] p
    cdef cnp.float64_t[:,:,:] boxes
    if fit_func == "fit_gaus":
        z_opt = kwargs["z_opt"]
        sigma = kwargs["sigma"]
        w = hanoi.fit_gaus(z=z, z_opt = z_opt, sigma = sigma)
    if fit_func == "fit_multimodal":
        sigma = kwargs["sigma"]
        p = kwargs["p"]
        z_opt = kwargs["z_opt"]
        w = hanoi.fit_multimodal(sigma=sigma, p=kwargs.get('p'), z_opt=z_opt, z=z)
    if fit_func == "fit_neutral":
        w = hanoi.fit_neutral(z=z)
    if fit_func == "fit_step":
        boxes = kwargs["boxes"]
        w = hanoi.fit_step(z=z, boxes=boxes)
    
    # Reproduction
    cdef cnp.ndarray[cnp.float64_t, ndim = 2] genotype_next
    cdef cnp.ndarray[cnp.int64_t, ndim = 1] n_offspring
    genotype_next, n_offspring = hanoi.sim_reproduction(popsize, genotype, w)
    #if n_offspring.sum() == 0:
    #    raise RuntimeError("No individuals survived")
    if mut_rate > 0:
        genotype_next = hanoi.sim_mutation(genotype=genotype_next, mut_rate=mut_rate) # This still returns nans if it is nans
    return hanoi.Generation(genotype = genotype, 
                      genotype_next = genotype_next,
                      breed_val = A, 
                      phenotype = z, 
                      fitness = w,
                      n_offspring = n_offspring
                     )

cpdef object sim_generations_c(int n_gen, object mut_effect, cnp.ndarray[cnp.float64_t, ndim=2] genotype, double mut_rate, str fit_func, cnp.ndarray[cnp.float64_t, ndim = 1] mean_0, cnp.ndarray[cnp.float64_t, ndim = 2] cov_0, dict kwargs = {}):
    cdef list generations_list = []
    cdef cnp.ndarray[cnp.float64_t, ndim = 2] genotype_cur = genotype
    cdef int i
    if n_gen == 0:
        i = 0
        while not (np.all(np.isin(genotype_cur, [0,2])) or np.all(np.isnan(genotype_cur))):
            #while np.any(genotype_cur == 1) or not np.isnan(genotype_cur[0,0]):
            generations_list.append(sim_generation_c(mut_effect = mut_effect, 
                                                     genotype = genotype_cur, 
                                                     mut_rate = mut_rate, 
                                                     fit_func = fit_func, 
                                                     mean_0 = mean_0,
                                                     cov_0 = cov_0,
                                                     kwargs = kwargs))
            genotype_cur = generations_list[-1].genotype_next
            #genotype_cur = sim_generation_c(mut_effect = mut_effect, 
            #                                         genotype = genotype_cur, 
            #                                         mut_rate = mut_rate, 
            #                                         fit_func = fit_func, 
            #                                         mean_0 = mean_0,
            #                                         cov_0 = cov_0,
            #                                         kwargs = kwargs).genotype_next
            i+=1
    else:
        for i in range(n_gen):
            if np.all(np.isnan(genotype_cur)):
                break
            generations_list.append(sim_generation_c(mut_effect = mut_effect, 
                                                     genotype = genotype_cur, 
                                                     mut_rate = mut_rate, 
                                                     fit_func = fit_func, 
                                                     mean_0 = mean_0,
                                                     cov_0 = cov_0,
                                                     kwargs = kwargs))
            genotype_cur = generations_list[-1].genotype_next
    cdef object generations = hanoi.Generations(genotype = np.array([generation.genotype for generation in generations_list]), 
                                                genotype_next = np.array([generation.genotype_next for generation in generations_list]), 
                                                breed_val = np.array([generation.breed_val for generation in generations_list]),
                                                phenotype = np.array([generation.phenotype for generation in generations_list]),
                                                fitness = np.array([generation.fitness for generation in generations_list]),
                                                n_offspring = np.array([generation.n_offspring for generation in generations_list]),
                                                n_gen = i
                                               )
    return generations


Content of stderr:
In file included from /home/jishigohoka/miniforge3/lib/python3.12/site-packages/numpy/_core/include/numpy/ndarraytypes.h:1909,
                 from /home/jishigohoka/miniforge3/lib/python3.12/site-packages/numpy/_core/include/numpy/ndarrayobject.h:12,
                 from /home/jishigohoka/miniforge3/lib/python3.12/site-packages/numpy/_core/include/numpy/arrayobject.h:5,
                 from /home/jishigohoka/.cache/ipython/cython/_cython_magic_89ffe48fc4baa3e6091701f20b41e202c3509b8360ac56c81375689a02a4e3b8.c:1156:
/home/jishigohoka/miniforge3/lib/python3.12/site-packages/numpy/_core/include/numpy/npy_1_7_deprecated_api.h:17:2: warning: #warning "Using deprecated NumPy API, disable it with " "#define NPY_NO_DEPRECATED_API NPY_1_7_API_VERSION" [-Wcpp]
   17 | #warning "Using deprecated NumPy API, disable it with " \
      |  ^~~~~~~
In function ‘__Pyx_PyLong_From_int’,
    inlined from ‘__pyx_f_78_cython_magic_89ffe48fc4baa3e6091701f20b41e202c3509b8360ac56c8137568

In [4]:
n = 1
L = 1000
N = 100

In [5]:
M = hanoi.MutEffect(mean = np.ones((n, L), dtype = np.float64), 
                    var = np.ones((n, L), dtype = np.float64), 
                    cov = np.zeros((math.comb(n, 2), L), dtype = np.float64)
                   )

In [6]:
gt = np.ones((L, N), dtype = np.float64)

In [7]:
mean_0 = np.array([0], dtype = float)
varcov_0 = np.array([[0.1]])

In [12]:
%%time
gens = hanoi.sim_generations(n_gen = 10, mut_effect = M, genotype = gt, fit_func = "fit_gaus", mut_rate = 0.0, mean_0 = mean_0, cov_0 = varcov_0, z_opt = np.array([L]), sigma = 10)
print(gens.n_gen)

9
CPU times: user 36.1 ms, sys: 2.01 ms, total: 38.1 ms
Wall time: 37.4 ms


In [13]:
gens.genotype[-1]

array([[2., 2., 2., ..., 2., 2., 2.],
       [0., 0., 0., ..., 0., 0., 0.],
       [2., 2., 2., ..., 2., 2., 2.],
       ...,
       [0., 0., 0., ..., 0., 0., 0.],
       [0., 0., 0., ..., 0., 0., 0.],
       [0., 0., 0., ..., 0., 0., 0.]])

In [14]:
%%time
gens = sim_generations_c(n_gen = 0, mut_effect = M, genotype = gt, fit_func = "fit_gaus", mut_rate = 0.0, mean_0 = mean_0, cov_0 = varcov_0, kwargs = {"z_opt": np.array([L], dtype = float), "sigma": 10})
print(gens.n_gen)

NameError: name 'sim_generations_c' is not defined

In [15]:
gens.n_gen

9

In [16]:
hanoi.sim_reproduction(N, gt, np.random.uniform(0, 1, size = N))

(array([[2, 2, 0, ..., 0, 1, 1],
        [0, 1, 1, ..., 1, 1, 1],
        [1, 0, 1, ..., 2, 0, 2],
        ...,
        [1, 1, 0, ..., 2, 1, 1],
        [2, 2, 2, ..., 1, 1, 2],
        [0, 2, 0, ..., 1, 2, 0]]),
 array([1, 4, 1, 0, 2, 3, 3, 1, 3, 5, 1, 1, 1, 1, 5, 1, 3, 3, 2, 4, 3, 3,
        6, 5, 2, 3, 3, 1, 1, 0, 1, 2, 2, 0, 3, 3, 1, 1, 0, 0, 0, 2, 1, 0,
        0, 2, 1, 1, 2, 3, 1, 0, 1, 0, 4, 5, 3, 0, 4, 5, 5, 1, 3, 0, 2, 3,
        4, 0, 3, 0, 0, 0, 0, 0, 4, 5, 6, 0, 1, 0, 0, 4, 1, 4, 3, 2, 3, 1,
        3, 5, 6, 1, 0, 3, 1, 4, 1, 1, 0, 0]))

In [13]:
sim = hanoi.sim_generation(mut_effect = M, genotype = gt, fit_func = "fit_gaus", mut_rate = 0, mean_0 = mean_0, cov_0 = varcov_0, z_opt = np.array([0], dtype = float), sigma= 1)

In [17]:
%load_ext line_profiler

In [18]:
%lprun -f hanoi.sim_reproduction hanoi.sim_generations(n_gen = 0, mut_effect = M, genotype = gt, fit_func = "fit_gaus", mut_rate = 0, mean_0 = mean_0, cov_0 = varcov_0, z_opt = np.array([0], dtype = float), sigma= 1)

Timer unit: 1e-09 s

Total time: 0.0292591 s
File: /home/jishigohoka/Dropbox/work/projects/python/packages/hanoi/hanoi/hanoi.py
Function: sim_reproduction at line 269

Line #      Hits         Time  Per Hit   % Time  Line Contents
   269                                           def sim_reproduction(popsize, genotype, fitness ):
   270                                               # number of offspring per genotype of each genotype
   271        12      16986.0   1415.5      0.1      n_loci = genotype.shape[0]
   272        12      49491.0   4124.2      0.2      if fitness.sum() == 0:
   273                                                   genotype_next = np.full((n_loci, popsize), np.nan)
   274                                                   n_offspring = np.zeros(popsize, dtype = int)
   275                                                   #raise RuntimeError("Fitness of all individuals is 0")
   276                                                   return genotype_next, n_offsp

In [19]:
hanoi.sim_reproduction??

Signature: hanoi.sim_reproduction(popsize, genotype, fitness)
Docstring: <no docstring>
Source:   
def sim_reproduction(popsize, genotype, fitness ):
    # number of offspring per genotype of each genotype
    n_loci = genotype.shape[0]
    if fitness.sum() == 0:
        genotype_next = np.full((n_loci, popsize), np.nan)
        n_offspring = np.zeros(popsize, dtype = int)
        #raise RuntimeError("Fitness of all individuals is 0")
        return genotype_next, n_offspring
    n_offspring = np.random.multinomial(n = 2 * popsize, pvals = fitness/fitness.sum())

    # index of 2N parents
    sam = np.repeat(range(popsize), n_offspring)
    # Shuffle the 2N parents
    np.random.shuffle(sam)
    # Reshape parents so that they are in pairs
    sam = sam.reshape(popsize, 2)
    # Mating
    #genotype_next = np.random.binomial(1, genotype[:,sam]/2 ).sum(axis = 2)
    genotype_next = (np.random.rand(*genotype[:, sam].shape) < genotype[:, sam] / 2).sum(axis=2)
    return genotype_next, n_of

In [28]:
np.__version__


'2.0.2'

In [2]:
hanoi.sim_reproduction_c?

Signature:      hanoi.sim_reproduction_c(popsize, genotype, fitness)
Call signature: hanoi.sim_reproduction_c(*args, **kwargs)
Type:           cython_function_or_method
String form:    <cyfunction sim_reproduction_c at 0x74449eac68c0>
Docstring:      <no docstring>

In [4]:
hanoi.sim_generation??

Signature:
hanoi.sim_generation(
    mut_effect,
    genotype,
    mut_rate,
    fit_func,
    mean_0,
    cov_0,
    **kwargs,
)
Docstring: <no docstring>
Source:   
def sim_generation(mut_effect, genotype, mut_rate, fit_func, mean_0, cov_0, **kwargs):
    popsize = genotype.shape[1]
    
    # Breeding value
    A = comp_breed_val(mut_effect=mut_effect, genotype=genotype)
    
    # Phenotype
    z = sim_pheno(breed_val=A, mean_0=mean_0, cov_0 = cov_0)
    
    # Phenotype to fitness
    if fit_func == "fit_gaus":
        w = fit_gaus(z=z, **kwargs)
    if fit_func == "fit_multimodal":
        w = fit_multimodal(sigma=sigma, p=kwargs.get('p'), z_opt=z_opt, z=z)
    if fit_func == "fit_neutral":
        w = fit_neutral(z=z)
    if fit_func == "fit_step":
        ranges = kwargs.get('boxes')
        w = fit_step(z=z, boxes=ranges)
    
    # Reproduction
    genotype_next, n_offspring = sim_reproduction_c(popsize=popsize, genotype=genotype, fitness=w)
    #if n_offspring.sum() == 0:
  

In [54]:
%%cython
import numpy as np
cimport numpy as cnp
cimport cython


def sim_mutation(cnp.float64_t[:,:] genotype, double mut_rate):
    # Compute transition matrix
    cdef cnp.float64_t[:,:] transition_matrix = np.array([[(1 - mut_rate)**2, 2 * mut_rate * (1 - mut_rate), mut_rate**2],
                                  [mut_rate * (1 - mut_rate), (1 - mut_rate)**2 + mut_rate**2, mut_rate * (1 - mut_rate)],
                                  [mut_rate**2, 2 * mut_rate * (1 - mut_rate), (1 - mut_rate)**2]])

    # Count the number of entries in the genotype matrix for each of the 3 genotypes
    cdef cnp.int64_t[:] n_sites_genotype
    n_sites_genotype = np.zeros(3, dtype = np.int64)
    cdef int i, j, k
    cdef int n_loci = genotype.shape[0]
    cdef int n_ind = genotype.shape[1]
    for i in range(n_loci): # loop over three possible genotypes
        for j in range(n_ind):
            if genotype[i,j] == 0:
                n_sites_genotype[0] += 1
            elif genotype[i,j] == 1:
                n_sites_genotype[1] += 1
            elif genotype[i,j] == 2:
                n_sites_genotype[2] += 1


    return np.asarray(n_sites_genotype)
    

Content of stderr:
In file included from /home/jishigohoka/miniforge3/lib/python3.12/site-packages/numpy/_core/include/numpy/ndarraytypes.h:1909,
                 from /home/jishigohoka/miniforge3/lib/python3.12/site-packages/numpy/_core/include/numpy/ndarrayobject.h:12,
                 from /home/jishigohoka/miniforge3/lib/python3.12/site-packages/numpy/_core/include/numpy/arrayobject.h:5,
                 from /home/jishigohoka/.cache/ipython/cython/_cython_magic_c5172e558c0b290bec7caf82c84061acc9d74cb10e2fb21faf212750d3e2af8e.c:1155:
/home/jishigohoka/miniforge3/lib/python3.12/site-packages/numpy/_core/include/numpy/npy_1_7_deprecated_api.h:17:2: warning: #warning "Using deprecated NumPy API, disable it with " "#define NPY_NO_DEPRECATED_API NPY_1_7_API_VERSION" [-Wcpp]
   17 | #warning "Using deprecated NumPy API, disable it with " \
      |  ^~~~~~~

In [55]:
sim_mutation(gt, 0.1)

array([8, 3, 1])

In [8]:
gt = np.array([[0, 1, 0, 0],
               [0, 1, 0, 0],
               [0, 1, 0, 2]], dtype = np.float64)

In [9]:
np.array([(gt == i).sum() for i in range(3)])

array([8, 3, 1])

In [10]:
np.zeros(3, dtype = np.int64)

array([0, 0, 0])

In [11]:
gt==0

array([[ True, False,  True,  True],
       [ True, False,  True,  True],
       [ True, False,  True, False]])

In [57]:
{i : np.where(gt == i) for i in range(3)}


{0: (array([0, 0, 0, 1, 1, 1, 2, 2]), array([0, 2, 3, 0, 2, 3, 0, 2])),
 1: (array([0, 1, 2]), array([1, 1, 1])),
 2: (array([2]), array([3]))}

In [82]:
%%cython
import numpy as np
cimport numpy as cnp
cimport cython

cpdef sim_generations(int n_gen, 
                      object mut_effect, 
                      cnp.float64_t[:,:] genotype, 
                      double mut_rate, 
                      str fit_func, 
                      cnp.float64_t[:] mean_0, 
                      cnp.float64_t [:,:] cov_0, 
                      dict kwargs):
    cdef list generations_list = []
    cdef cnp.float64_t[:, :] genotype_cur = genotype
    if n_gen == 0:
       i = 0
       while not (np.all(np.isin(genotype_cur, [0,2])) or np.all(np.isnan(genotype_cur))):
            


    return "hello"



Content of stderr:
In file included from /home/jishigohoka/miniforge3/lib/python3.12/site-packages/numpy/_core/include/numpy/ndarraytypes.h:1909,
                 from /home/jishigohoka/miniforge3/lib/python3.12/site-packages/numpy/_core/include/numpy/ndarrayobject.h:12,
                 from /home/jishigohoka/miniforge3/lib/python3.12/site-packages/numpy/_core/include/numpy/arrayobject.h:5,
                 from /home/jishigohoka/.cache/ipython/cython/_cython_magic_463b020eac2fc01e5a2d08cd810315e2ec691ea9767a77818dcee0dcba966063.c:1162:
/home/jishigohoka/miniforge3/lib/python3.12/site-packages/numpy/_core/include/numpy/npy_1_7_deprecated_api.h:17:2: warning: #warning "Using deprecated NumPy API, disable it with " "#define NPY_NO_DEPRECATED_API NPY_1_7_API_VERSION" [-Wcpp]
   17 | #warning "Using deprecated NumPy API, disable it with " \
      |  ^~~~~~~

In [3]:
gt = np.array([[0, 1, 0, 0],
               [0, 1, 0, 0],
               [0, 1, 0, 2]], dtype = np.float64)

In [13]:
M = hanoi.MutEffect(mean = np.array(([[1,1,1]])), cov = np.array([[]]), var = np.array([[1, 1, 1]]))

In [15]:
%load_ext line_profiler

The line_profiler extension is already loaded. To reload it, use:
  %reload_ext line_profiler


In [22]:
%lprun -f hanoi.sim_generation  hanoi.sim_generation(mut_effect = M, genotype = gt, mut_rate = 0.0, mean_0 = np.array([0], dtype = float), cov_0 = np.array([[1]], dtype = float), fit_func = "fit_neutral", kwargs = dict())

Timer unit: 1e-09 s

Total time: 0.000507446 s
File: /home/jishigohoka/Dropbox/work/projects/python/packages/hanoi/hanoi/hanoi.py
Function: sim_generation at line 140

Line #      Hits         Time  Per Hit   % Time  Line Contents
   140                                           def sim_generation(mut_effect, genotype, mut_rate, fit_func, mean_0, cov_0, **kwargs):
   141         1       2750.0   2750.0      0.5      popsize = genotype.shape[1]
   142                                               
   143                                               # Breeding value
   144         1     132016.0 132016.0     26.0      A = comp_breed_val(mut_effect=mut_effect, genotype=genotype)
   145                                               
   146                                               # Phenotype
   147                                               #z = sim_pheno(breed_val=A, mean_0=mean_0, cov_0 = cov_0)
   148         1      90713.0  90713.0     17.9      z = sim_pheno_c(breed_val=A, me

In [6]:
%lprun -f hanoi.sim_generations  hanoi.sim_generations(n_gen = 0, mut_effect = M, genotype = gt, mut_rate = 0.0, mean_0 = np.array([0], dtype = float), cov_0 = np.array([[1]], dtype = float), fit_func = "fit_neutral", kwargs = dict())

UsageError: Line magic function `%lprun` not found.


In [18]:
A = hanoi.comp_breed_val(M, gt)

In [5]:
%lprun -f hanoi.sim_pheno_c  hanoi.sim_pheno_c(breed_val = A, mean_0 = np.array([0], dtype = float), cov_0 = np.array([[1]], dtype = float))

UsageError: Line magic function `%lprun` not found.


In [62]:
def myfunc(n):
    for i in range(n):
        chol=jnp.linalg.cholesky(A.mean + A.varcov)
        z_std = np.random.standard_normal((A.N, A.n))
        z = np.einsum('ijk,ik->ij', chol, z_std) + np.array([1])
        #z = jnp.matmul(chol, z_std[..., None])[..., 0] + np.array([1])




In [82]:
def myfunc1(n):
    for i in range(n):
        z = np.random.normal(A.mean[:,0], A.varcov[:, 0, 0])

In [77]:
A.mean[:,0]

array([0. , 1.5, 0. , 1. ])

In [60]:
myfunc(100_000)

In [83]:
%lprun -f myfunc myfunc(100_000)

Timer unit: 1e-09 s

Total time: 1.52025 s
File: /tmp/ipykernel_265306/1231169443.py
Function: myfunc at line 1

Line #      Hits         Time  Per Hit   % Time  Line Contents
     1                                           def myfunc(n):
     2    100001   27075378.0    270.8      1.8      for i in range(n):
     3    100000  637101882.0   6371.0     41.9          chol=jnp.linalg.cholesky(A.mean + A.varcov)
     4    100000   87484320.0    874.8      5.8          z_std = np.random.standard_normal((A.N, A.n))
     5    100000  768587877.0   7685.9     50.6          z = np.einsum('ijk,ik->ij', chol, z_std) + np.array([1])
     6                                                   #z = jnp.matmul(chol, z_std[..., None])[..., 0] + np.array([1])

In [84]:
%lprun -f myfunc1 myfunc1(100_000)

Timer unit: 1e-09 s

Total time: 0.813945 s
File: /tmp/ipykernel_265306/103944918.py
Function: myfunc1 at line 1

Line #      Hits         Time  Per Hit   % Time  Line Contents
     1                                           def myfunc1(n):
     2    100001   24890793.0    248.9      3.1      for i in range(n):
     3    100000  789054461.0   7890.5     96.9          z = np.random.normal(A.mean[:,0], A.varcov[:, 0, 0])

In [92]:
A.mean[0,:]

array([0.])

In [46]:
%%time
for i in range(100_000):
    chol=scipy.linalg.cholesky(np.array([[1]]) + A.varcov)
    z_std = np.random.standard_normal((A.N, A.n))
    z = np.einsum('ijk,ik->ij', chol, z_std) + np.array([1])


CPU times: user 4.68 s, sys: 131 ms, total: 4.81 s
Wall time: 4.66 s


In [68]:
gt


array([[0., 1., 0., 0.],
       [0., 1., 0., 0.],
       [0., 1., 0., 2.]])

In [3]:
gt = np.array([[0, 1, 2],
               [1, 2, 0],
               [2, 0, 1]], dtype = np.float64)

## Optimise sim_reproduction

In [12]:
gt = np.array([[0, 1, 1, 2, 2],
               [0, 1, 2, 0, 2],
              [0,0,0,0,0]])

In [13]:
gt

array([[0, 1, 1, 2, 2],
       [0, 1, 2, 0, 2],
       [0, 0, 0, 0, 0]])

In [14]:
n_offspring = np.random.multinomial(n = 2 * 5, pvals = [0.1,  0.2, 0.3, 0.4, 0.5]) 

In [15]:
n_offspring

array([0, 2, 4, 4, 0])

In [16]:
sam = np.repeat(range(5), n_offspring)
np.random.shuffle(sam)
sam = sam.reshape(5,2)
sam

array([[3, 2],
       [1, 3],
       [2, 3],
       [1, 2],
       [3, 2]])

In [321]:
np.random.binomial(n = 1, p = gt[:, sam]/2).sum(axis = 2)

array([[1, 1, 2, 1, 2],
       [1, 0, 1, 2, 0],
       [0, 0, 0, 0, 0]])

In [84]:
np.random.binomial??

Signature: np.random.binomial(n, p, size=None)
Docstring:
binomial(n, p, size=None)

Draw samples from a binomial distribution.

Samples are drawn from a binomial distribution with specified
parameters, n trials and p probability of success where
n an integer >= 0 and p is in the interval [0,1]. (n may be
input as a float, but it is truncated to an integer in use)

.. note::
    New code should use the `~numpy.random.Generator.binomial`
    method of a `~numpy.random.Generator` instance instead;
    please see the :ref:`random-quick-start`.

Parameters
----------
n : int or array_like of ints
    Parameter of the distribution, >= 0. Floats are also accepted,
    but they will be truncated to integers.
p : float or array_like of floats
    Parameter of the distribution, >= 0 and <=1.
size : int or tuple of ints, optional
    Output shape.  If the given shape is, e.g., ``(m, n, k)``, then
    ``m * n * k`` samples are drawn.  If size is ``None`` (default),
    a single value is returned 

In [58]:
%%cython
import numpy as np
cimport numpy as cnp
cimport cython 
from hanoi import MutEffect, BreedVal, Generation, Generations


def sim_generation(mut_effect, genotype, mut_rate, fit_func, mean_0, cov_0, **kwargs):
    popsize = genotype.shape[1]


Content of stderr:
In file included from /home/jishigohoka/miniforge3/lib/python3.12/site-packages/numpy/_core/include/numpy/ndarraytypes.h:1909,
                 from /home/jishigohoka/miniforge3/lib/python3.12/site-packages/numpy/_core/include/numpy/ndarrayobject.h:12,
                 from /home/jishigohoka/miniforge3/lib/python3.12/site-packages/numpy/_core/include/numpy/arrayobject.h:5,
                 from /home/jishigohoka/.cache/ipython/cython/_cython_magic_b3428db6d6b0e000804b69d332ff380ca4745bd03a45119d71bb185c8dc35f69.c:1156:
/home/jishigohoka/miniforge3/lib/python3.12/site-packages/numpy/_core/include/numpy/npy_1_7_deprecated_api.h:17:2: warning: #warning "Using deprecated NumPy API, disable it with " "#define NPY_NO_DEPRECATED_API NPY_1_7_API_VERSION" [-Wcpp]
   17 | #warning "Using deprecated NumPy API, disable it with " \
      |  ^~~~~~~

In [20]:
np.random.normal?


Signature: np.random.normal(loc=0.0, scale=1.0, size=None)
Docstring:
normal(loc=0.0, scale=1.0, size=None)

Draw random samples from a normal (Gaussian) distribution.

The probability density function of the normal distribution, first
derived by De Moivre and 200 years later by both Gauss and Laplace
independently [2]_, is often called the bell curve because of
its characteristic shape (see the example below).

The normal distributions occurs often in nature.  For example, it
describes the commonly occurring distribution of samples influenced
by a large number of tiny, random disturbances, each with its own
unique distribution [2]_.

.. note::
    New code should use the `~numpy.random.Generator.normal`
    method of a `~numpy.random.Generator` instance instead;
    please see the :ref:`random-quick-start`.

Parameters
----------
loc : float or array_like of floats
    Mean ("centre") of the distribution.
scale : float or array_like of floats
    Standard deviation (spread or "width")

In [98]:
cProfile.run('hanoi.sim_generations(n_gen = 0, mut_effect=M, mut_rate=0, genotype=gt, mean_0 = np.array([0]), cov_0=np.array([[]]), fit_func="fit_neutral")' )

         1121 function calls (1116 primitive calls) in 0.004 seconds

   Ordered by: standard name

   ncalls  tottime  percall  cumtime  percall filename:lineno(function)
        4    0.000    0.000    0.000    0.000 <frozen _collections_abc>:341(__subclasshook__)
        4    0.000    0.000    0.000    0.000 <frozen abc>:117(__instancecheck__)
        6    0.000    0.000    0.000    0.000 <frozen abc>:121(__subclasscheck__)
        3    0.000    0.000    0.000    0.000 <frozen importlib._bootstrap>:1390(_handle_fromlist)
        1    0.000    0.000    0.003    0.003 <string>:1(<module>)
        5    0.000    0.000    0.000    0.000 _arraysetops_impl.py:833(_in1d)
       10    0.000    0.000    0.000    0.000 _arraysetops_impl.py:847(<genexpr>)
        5    0.000    0.000    0.000    0.000 _arraysetops_impl.py:981(_isin_dispatcher)
        5    0.000    0.000    0.000    0.000 _arraysetops_impl.py:986(isin)
       12    0.000    0.000    0.000    0.000 _methods.py:50(_sum)
       16  

In [99]:
cProfile.run?


Signature: cProfile.run(statement, filename=None, sort=-1)
Docstring:
Run statement under profiler optionally saving results in filename

This function takes a single argument that can be passed to the
"exec" statement, and an optional file name.  In all cases this
routine attempts to "exec" its first argument and gather profiling
statistics from the execution. If no file name is present, then this
function automatically prints a simple profiling report, sorted by the
standard name string (file/line/function-name) that is presented in
each line.
File:      ~/miniforge3/lib/python3.12/cProfile.py
Type:      function

In [91]:
p = pstats.Stats("prof")


NameError: name 'prof' is not defined

In [84]:
%%cython
import numpy as np
cimport numpy as cnp
cimport cython
import jax.numpy as jnp

def sim_pheno(object breed_val, cnp.float64_t[:] mean_0, cnp.float64_t[:,:] cov_0):
    cdef cnp.float64_t[:,:] z
    if breed_val.n == 1:
        z = np.random.normal(mean_0[0] + breed_val.mean, cov_0[0,0] + breed_val.var, (breed_val.N, breed_val.n))
        return np.asarray(z)
        #return z
    # Cholesky decomposition of the covariance matrices
    chol = jnp.linalg.cholesky(cov_0 + breed_val.varcov)
    # Standard normal sampling
    z_std = np.random.standard_normal((breed_val.N, breed_val.n))
    # Convert N vectors of n standard normal variables to the N phenotype values in n dimensions
    z = np.einsum('ijk,ik->ij', chol, z_std) + mean_0
    return z


Content of stderr:
In file included from /home/jishigohoka/miniforge3/lib/python3.12/site-packages/numpy/_core/include/numpy/ndarraytypes.h:1909,
                 from /home/jishigohoka/miniforge3/lib/python3.12/site-packages/numpy/_core/include/numpy/ndarrayobject.h:12,
                 from /home/jishigohoka/miniforge3/lib/python3.12/site-packages/numpy/_core/include/numpy/arrayobject.h:5,
                 from /home/jishigohoka/.cache/ipython/cython/_cython_magic_acaa03b1dbd63206baf4cdfa25bbdeb64fe2868827e4d759b89dd06a0eb505fd.c:1173:
/home/jishigohoka/miniforge3/lib/python3.12/site-packages/numpy/_core/include/numpy/npy_1_7_deprecated_api.h:17:2: warning: #warning "Using deprecated NumPy API, disable it with " "#define NPY_NO_DEPRECATED_API NPY_1_7_API_VERSION" [-Wcpp]
   17 | #warning "Using deprecated NumPy API, disable it with " \
      |  ^~~~~~~

In [85]:
M = hanoi.MutEffect(mean = np.array(([[1,1,1]])), cov = np.array([[]]), var = np.array([[1, 1, 1]]))
gt = np.array([[0, 1, 0, 0],
               [0, 1, 0, 0],
               [0, 1, 0, 2]], dtype = np.float64)

A = hanoi.comp_breed_val(M, gt)

In [86]:
A.var[:,0]

array([0. , 1.5, 0. , 1. ])

In [87]:
sim_pheno(breed_val = A, 
             mean_0 = np.array([0], dtype = float), 
             cov_0 = np.array([[1]], dtype = float))

array([[ 1.11186262],
       [ 0.43083142],
       [-0.86220059],
       [ 2.20212098]])

In [56]:
%%time
res = []
for i in range(10):
    sim = sim_generations(n_gen = 0, 
                        mut_effect = M, 
                        genotype = gt, 
                        mean_0 = mu_0,
                        cov_0 = cov_0, 
                        fit_func =  "fit_neutral",
                        boxes = np.array([[[1], [math.inf]]]) ,
                        mut_rate = 0
                       )
    res.append([sim.n_gen, sim.allele_freq_last()[0]])
print({val: int((np.array(res)[:,1] == val).sum()) for val in [0, 1, np.nan]})

        

(4, 1)

In [21]:
np.random.normal(1, 1, (10,1))

array([[ 0.68398279],
       [ 1.56354774],
       [ 0.06877977],
       [-0.52059801],
       [ 1.26743548],
       [ 0.05740898],
       [ 1.60362742],
       [ 1.36379772],
       [ 0.94813156],
       [ 0.95235932]])